# Previous Code For Future Use

In [ ]:
%pip install -U langchain-google-genai

In [ ]:
import requests
from langchain.agents import initialize_agent, Tool
from langchain_google_genai import ChatGoogleGenerativeAI  # Gemini
# from langchain_ollama import ChatOllama  # Uncomment for Ollama

# ========================
# CONFIG
# ========================
GOOGLE_API_KEY=""
NEWSAPI_KEY = ""   # 👈 replace with your NewsAPI key

# ========================
# TOOL (Direct Fetch)
# ========================
def get_newsapi(query):
    """Fetch news from NewsAPI without caching"""
    url = f"https://newsapi.org/v2/everything?q={query}&apiKey={NEWSAPI_KEY}&pageSize=5"
    resp = requests.get(url).json()

    if "articles" in resp:
        return "\n".join(
            [f"- {a['title']} ({a['source']['name']})" for a in resp["articles"]]
        )
    return str(resp)

tools = [
    Tool(
        name="NewsAPI",
        func=lambda q: str(get_newsapi(q)),
        description="Fetch latest news articles about a topic from NewsAPI.org"
    )
]

# ========================
# LLM CHOICE
# ========================
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    api_key=GOOGLE_API_KEY   # Use the Python variable, not os.environ
)

# Cloud option
# llm = ChatOllama(model="llama3")  # Local option if Ollama is installed

# ========================
# AGENT
# ========================
agent = initialize_agent(tools, llm, agent="zero-shot-react-description", verbose=True, handle_parsing_errors=True)

# ========================
# TEST QUERY
# ========================
if __name__ == "__main__":
    query = "artificial intelligence in the job market"
    result = agent.run(f"Find the latest news about {query}")
    print("\n🔎 Agent Result:\n", result)



> Entering new AgentExecutor chain...
Thought: I need to use NewsAPI to find the latest news articles about AI and the job market.  I'll need to construct a suitable query.

Action: NewsAPI
Action Input: q=artificial+intelligence+job+market
Observation: - The Prompt: VCs Aren’t Happy About AI Founders Jumping Ship For Big Tech (Forbes)
- Amazon's AI-Powered AWS, Efficiency Gains, And Consumer Demand Fuel Bullish Q2 Outlook (Yahoo Entertainment)
- 3 Questions for Online Learning Pioneer Robert Ubell (Inside Higher Ed)
- GPT-5s Are Alive: Synthesis (Lesswrong.com)
- Trump Won’t Take Away Tesla’s Subsidies. Does That Make TSLA Stock a Safe Buy Here? (Barchart.com)
Thought:Thought: The NewsAPI results are somewhat mixed. Some articles mention AI in relation to business and technology, but not directly concerning its impact on the job market. I need a more specific query.

Action: NewsAPI
Action Input: q="artificial intelligence" AND ("job market" OR "employment" OR "jobs" OR "workforce")

In [ ]:
import requests
import pandas as pd

API_KEY = ""  # replace with your key

def collect_articles(query="future of work skills", category="business", max_results=50):
    url = "https://api.worldnewsapi.com/search-news"
    params = {
        "text": query,
        "language": "en",
        "api-key": API_KEY,
        "sort": "publish-time",
        "categories": category
    }

    response = requests.get(url, params=params)
    data = response.json()

    if "news" not in data:
        print("No results:", data)
        return pd.DataFrame()

    # Extract useful fields
    articles = []
    for art in data["news"][:max_results]:
        # Safely access 'source' and 'name' using .get()
        source_name = art.get("source", {}).get("name")
        articles.append({
            "title": art.get("title"),
            "url": art.get("url"),
            "published": art.get("publish_date"),
            "source": source_name,
            "categories": ", ".join(art.get("categories", [])),
            "summary": art.get("summary", "")
        })

    # Create dataframe
    df = pd.DataFrame(articles)
    return df


# Example usage
df = collect_articles("artificial intelligence skills", "business", max_results=20)

# Save dataset
df.to_csv("ai_skills_news.csv", index=False)
df.to_json("ai_skills_news.json", orient="records", indent=2)

print(f"✅ Collected {len(df)} articles and saved to CSV & JSON.")

✅ Collected 10 articles and saved to CSV & JSON.


In [ ]:
import requests
import pandas as pd

# Replace with your World News API key
API_KEY = ""
BASE_URL = "https://api.worldnewsapi.com/search-news"

def fetch_articles(query, language="en", limit=10):
    params = {
        "text": query,
        "language": language,
        "number": limit,
        "api-key": API_KEY  # some APIs require this in params, others in headers
    }

    headers = {
        "x-api-key": API_KEY   # World News API prefers this in headers
    }

    response = requests.get(BASE_URL, params=params, headers=headers)

    if response.status_code != 200:
        print("Error:", response.status_code, response.text)
        return []

    data = response.json()
    return data.get("news", [])

def build_dataset(queries, articles_per_query=10):
    all_articles = []

    for query in queries:
        articles = fetch_articles(query, limit=articles_per_query)
        for art in articles:
            all_articles.append({
                "id": art.get("id"),
                "title": art.get("title"),
                "summary": art.get("summary"),
                "text": art.get("text"),
                "url": art.get("url"),
                "image": art.get("image"),
                "publish_date": art.get("publish_date"),
                "authors": ", ".join(art.get("authors", [])),
                "category": art.get("category"),
                "source_country": art.get("source_country"),
                "sentiment": art.get("sentiment")
                # If API provides "keywords" or "tags", add here
            })

    df = pd.DataFrame(all_articles)
    return df


if __name__ == "__main__":
    # Example queries (you can customize for skills/business/etc.)
    queries = ["AI workforce", "Top Skills to Highlight on Your CV", "Software Development Trends", "programming language"]

    dataset = build_dataset(queries, articles_per_query=5)

    # Save to CSV
    dataset.to_csv("news_dataset.csv", index=False, encoding="utf-8")
    print("Dataset saved as news_dataset.csv")

    # Also save JSON if needed
    dataset.to_json("news_dataset.json", orient="records", indent=2, force_ascii=False)
    print("Dataset saved as news_dataset.json")


Dataset saved as news_dataset.csv
Dataset saved as news_dataset.json


# Create Dataset

Import libraries

In [ ]:
!pip install gnews pandas openai tqdm


In [ ]:
import openai

# Replace with your OpenAI API key
openai.api_key = ""


In [ ]:
import pandas as pd
from gnews import GNews
import time
from tqdm import tqdm

# ---------------------------
# Settings
# ---------------------------
#TOPICS = ["Artificial Intelligence", "Machine Learning", "Cloud Computing",
#          "GoLang", "Blockchain", "Data Science", "Cybersecurity"]

TOPICS = ["New programming languages"]
ARTICLES_PER_TOPIC = 50  # Adjust to get ~1000 records in total
OUTPUT_FILE = "skill_zero_shot_dataset.csv"

# Initialize GNews
google_news = GNews(language='en', country='US', max_results=ARTICLES_PER_TOPIC)

# List to store dataset
dataset = []

# ---------------------------
# Fetch Articles & Extract Skills
# ---------------------------
for topic in TOPICS:
    print(f"\nFetching articles for topic: {topic}")
    articles = google_news.get_news(topic)

    for article in tqdm(articles):
        title = article.get("title", "")
        summary = article.get("description", "")
        text = (title or "") + ". " + (summary or "")
        if not text.strip():
            continue

        # -----------------------
        # LLM Zero-Shot Skill Extraction
        # -----------------------
        prompt = f"""
        Extract all skills, programming languages, tools, and technologies from the text below.
        Return as a Python list of strings.
        Text: "{text}"
        """
        try:
            response = openai.ChatCompletion.create(
                model="gpt-4-turbo",
                messages=[{"role": "user", "content": prompt}],
                temperature=0
            )
            skills_str = response["choices"][0]["message"]["content"]
            skills = eval(skills_str)  # Convert string to list
        except Exception as e:
            skills = []

        # -----------------------
        # Append to dataset
        # -----------------------
        dataset.append({
            "text": text,
            "source": article.get("source", {}).get("name", ""),
            "published_at": article.get("publishedAt", ""),
            "topic": topic,
            "skills": ", ".join(skills)
        })

        # Avoid rate limits
        time.sleep(1)

# ---------------------------
# Save Dataset
# ---------------------------
df = pd.DataFrame(dataset)
df.to_csv(OUTPUT_FILE, index=False)
print(f"\n✅ Dataset saved as '{OUTPUT_FILE}' with {len(df)} records")



Fetching articles for topic: New programming languages


100%|██████████| 50/50 [00:50<00:00,  1.00s/it]


✅ Dataset saved as 'skill_zero_shot_dataset.csv' with 50 records


Programming & Software

*   **Python, JavaScript, GoLang, Rust, Java**
*   **Web Development, Backend, Frontend**
*   **DevOps, Cloud Computing, Kubernetes, Docker**
*   Software Architecture, Microservices

AI & Data

*  Artificial Intelligence, Machine Learning, Deep Learning
*  Data Science, Data Engineering, Big Data, Analytics
*  Natural Language Processing, Computer Vision, Reinforcement Learning
*  Generative AI, LLMs, Prompt Engineering

Cybersecurity & Networking

*  Cybersecurity, Network Security, Ethical Hacking
*  Blockchain, Cryptography, Web3, Smart Contracts
*  Cloud Security, Zero Trust, Identity Management

Business & Product

*  Product Management, Agile, Scrum, DevOps Culture
*  Digital Transformation, SaaS, Fintech, E-Commerce
*  Project Management, Business Intelligence

Emerging Technologies

*  Quantum Computing, Edge Computing, IoT
*  AR/VR, Metaverse, Digital Twins
*  LangChain, AutoGen, Vector Databases, MLOps
*  Robotic Process Automation (RPA), Autonomous Systems

Healthcare & BioTech

*  HealthTech, Genomics, Bioinformatics
*  Medical Imaging, Wearables, AI in Healthcare

Cloud Providers / Platforms

*  AWS, Azure, Google Cloud, OpenStack
*  Terraform, Ansible, Jenkins, CI/CD

# Final 2 methods get articles from Gnews and RSS Feed

In [ ]:
import requests
import pandas as pd
from newspaper import Article
import time
import google.generativeai as genai
import json
import re

# ---------------------------
# Settings
# ---------------------------
API_KEY = ""  # GNews API key
GEN_API_KEY = ""  # Gemini API key
TOPICS = ["cyber security"]
ARTICLES_PER_TOPIC = 10
OUTPUT_CSV = "gnews_skills.csv"

genai.configure(api_key=GEN_API_KEY)

dataset = []

# ---------------------------
# Helper functions
# ---------------------------
def parse_json_skills(skills_str):
    """Parse JSON list from possibly messy model output"""
    skills_str = skills_str.strip()
    if not skills_str:
        return []
    try:
        return [s.strip() for s in json.loads(skills_str) if s.strip()]
    except json.JSONDecodeError:
        match = re.search(r'\[.*?\]', skills_str, re.DOTALL)
        if match:
            try:
                return [s.strip() for s in json.loads(match.group()) if s.strip()]
            except:
                return []
        return []

def extract_skills_gemini(text, retries=2):
    """Extract skills using Gemini 2.5 Flash"""
    prompt = f"""
    You are a strict skill extraction system.

    Your task:
    Extract ONLY skills that can be added directly to a resume.

    Allowed categories (MUST be explicitly mentioned in the text):
    - Technical skills
    - Programming languages (e.g., Python, JavaScript)
    - Frameworks and libraries (e.g., React, TensorFlow)
    - Cybersecurity tools, techniques, standards (e.g., Nmap, MITRE ATT&CK, ISO 27001)
    - Cloud technologies and services (AWS, Azure, GCP)
    - DevOps tools (Docker, Kubernetes, Jenkins)
    - Databases (MySQL, MongoDB, Redis)
    - AI/ML tools and models (PyTorch, LangChain, LLaMA)
    - Networking technologies (BGP, VPN, Zero Trust)
    - Certifications (CEH, CISSP, CCNA)
    - Any technology or tool that could appear in a "Skills" section on a resume

    STRICT RULES:
    1. Extract only items explicitly stated in the text. NO guessing.
    2. Exclude soft skills (e.g., communication, leadership, teamwork).
    3. Exclude job roles (e.g., analyst, engineer, researcher).
    4. Exclude responsibilities or tasks (e.g., monitoring networks, analyzing logs).
    5. Exclude generic words (e.g., “security”, “cloud”, “data”, “infrastructure”).
    6. Exclude company names and products (e.g., Google, Meta, OpenAI).
    7. Exclude long phrases; keep skills concise and standardized.
    8. Remove duplicates and ensure clean capitalization.
    9. Output MUST be ONLY a valid JSON list of strings with no explanation.
    10. If no resume-appropriate skills are found, return [].

    Example output:
    ["Python", "Kubernetes", "AWS Lambda", "Threat Intelligence", "ISO 27001"]

    Artical:
    "{text}"
    """

    model = genai.GenerativeModel("gemini-2.5-flash")

    delay = 1

    for attempt in range(retries + 1):
        try:
            response = model.generate_content(prompt)
            skills = parse_json_skills(response.text)
            if skills:
                return skills
        except Exception as e:
            print(f"⚠️ Gemini extraction failed: {e}")

        time.sleep(delay)
        delay *= 3
    return []

# ---------------------------
# Fetch & process articles
# ---------------------------
for topic in TOPICS:
    print(f"\nFetching articles for topic: {topic}")
    url = f"https://gnews.io/api/v4/search?q={topic}&token={API_KEY}&lang=en&max={ARTICLES_PER_TOPIC}"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        print(f"Error fetching API: {e}")
        continue

    for art in data.get("articles", []):
        title = art.get("title")
        description = art.get("description")
        article_url = art.get("url")
        source = art.get("source", {}).get("name", "")
        published_at = art.get("publishedAt", "")

        # Fetch full text with fallback
        try:
            time.sleep(1)
            a = Article(article_url)
            a.download()
            a.parse()
            full_text = a.text.strip() or description
        except:
            full_text = description or ""

        # Extract skills
        skills = extract_skills_gemini(full_text)

        dataset.append({
            "topic": topic,
            "title": title,
            "url": article_url,
            "source": source,
            "published_at": published_at,
            "summary": full_text[:500],  # optional: first 500 chars
            "skills": ", ".join(skills)
        })

# ---------------------------
# Save CSV
# ---------------------------
df = pd.DataFrame(dataset)
df.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ CSV saved: {OUTPUT_CSV} ({len(df)} articles)")



Fetching articles for topic: cyber security
⚠️ Gemini extraction failed: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
⚠️ Gemini extraction failed: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))

✅ CSV saved: gnews_skills.csv (10 articles)


In [ ]:
%pip install newspaper3k lxml_html_clean

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 35.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 8.2 MB/s eta 0:00:00
  Created wheel for tinysegmenter: filename=tinysegmenter-0.3-py3-none-any.whl size=13540 sha256=df6c46c9b321fc63283817d29b15d1a2fe5dd1dfd1307409608c32962dda240a
  Stored in directory: /root/.cache/pip/wheels/a5/91/9f/00d66475960891a64867914273fcaf78df6cb04d905b104a2a
  Created wheel for feedfinder2: filename=feedfinder2-0.0.4-py3-none-any.whl size=3341 sha256=1060b777724b6fa6d1ed16622f55e35472b5932367542a337a32165762c0a7fa
  Stored in directory: /root/.cache/pip/wheels/9f/9f/fb/364871d7426d3cdd4d293dcf7e53d97f16

In [ ]:
import requests
import pandas as pd
from newspaper import Article
import time
import google.generativeai as genai
import json
import re
from tqdm import tqdm

# -------------------------
# API Keys & Settings
# -------------------------
GNEWS_API_KEY = ""
GEN_API_KEY = ""

TOPICS = [
    "cybersecurity", "AI", "machine learning", "cloud computing",
    "programming", "web development", "data engineering"
]
ARTICLES_PER_TOPIC = 50  # ~50 articles per topic -> ~1000 total
OUTPUT_CSV = "skills_dataset.csv"

genai.configure(api_key=GEN_API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")

def clean_text(text):
    if not text:
        return ""
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip()

def get_article_text(url):
    try:
        a = Article(url)
        a.download()
        a.parse()
        return clean_text(a.text)
    except:
        return ""

def parse_json_skills(skills_str):
    skills_str = skills_str.strip()
    if not skills_str:
        return []
    try:
        return [s.strip() for s in json.loads(skills_str) if s.strip()]
    except json.JSONDecodeError:
        match = re.search(r'\[.*?\]', skills_str, re.DOTALL)
        if match:
            try:
                return [s.strip() for s in json.loads(match.group()) if s.strip()]
            except:
                return []
        return []

def extract_skills(text, retries=2):
    prompt = f"""
    Extract ONLY technical skills, tools, platforms, programming languages,
    cybersecurity terms, or technologies explicitly mentioned in the text.
    Return ONLY a valid JSON list of strings.

    Text:
    {text}
    """
    delay = 1
    for _ in range(retries + 1):
        try:
            response = model.generate_content(prompt)
            skills = parse_json_skills(response.text)
            if skills:
                return skills
        except Exception as e:
            print(f"⚠️ Gemini extraction failed: {e}")
            time.sleep(delay)
            delay *= 2
    return []

def tokenize_and_label(text, skills):
    tokens = text.split()
    labels = ["O"] * len(tokens)

    for skill in skills:
        skill_tokens = skill.split()
        for i in range(len(tokens) - len(skill_tokens) + 1):
            if tokens[i:i+len(skill_tokens)] == skill_tokens:
                labels[i] = "B-skill"
                for j in range(1, len(skill_tokens)):
                    labels[i+j] = "I-skill"
    return tokens, labels


In [ ]:
dataset = []

for topic in TOPICS:
    print(f"\nFetching articles for: {topic}")
    url = f"https://gnews.io/api/v4/search?q={topic}&token={GNEWS_API_KEY}&lang=en&max={ARTICLES_PER_TOPIC}"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        print(f"Error fetching articles: {e}")
        continue

    for art in tqdm(data.get("articles", []), desc=f"Processing {topic}"):
        title = art.get("title", "")
        summary = art.get("description", "")
        url = art.get("url", "")
        full_text = get_article_text(url) or summary

        text = f"{title}. {full_text}"  # combine title + summary
        skills = extract_skills(text)
        if not skills:
            continue  # skip if no skills extracted

        tokens, labels = tokenize_and_label(text, skills)
        for t, l in zip(tokens, labels):
            dataset.append({
                "topic": topic,
                "title": title,
                "token": t,
                "label": l
            })
df = pd.DataFrame(dataset)
df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Token-level BIO dataset saved: {OUTPUT_CSV} ({len(df)} tokens)")



Fetching articles for: cybersecurity


Processing cybersecurity: 100%|██████████| 10/10 [01:22<00:00,  8.21s/it]



Fetching articles for: AI


Processing AI: 100%|██████████| 10/10 [01:37<00:00,  9.79s/it]



Fetching articles for: machine learning


...

⚠️ Gemini extraction failed: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 10, model: gemini-2.5-flash
Please retry in 16.585851492s.


Processing web development: 100%|██████████| 10/10 [02:20<00:00, 14.01s/it]



Fetching articles for: data engineering


Processing data engineering: 100%|██████████| 10/10 [01:22<00:00,  8.27s/it]

✅ Token-level BIO dataset saved: skills_dataset.csv (33301 tokens)


In [ ]:
import requests
import pandas as pd
from newspaper import Article
import time
import google.generativeai as genai
import json
import re
from tqdm import tqdm

# -------------------------
# API Keys & Settings
# -------------------------
GNEWS_API_KEY = ""

GEMINI_KEYS = [
    "",
    "",
    "",
    ""
]

current_key = 0

genai.configure(api_key=GEMINI_KEYS[current_key])
model = genai.GenerativeModel("gemini-2.5-flash")

def switch_key():
    global current_key, model
    current_key = (current_key + 1) % len(GEMINI_KEYS)
    genai.configure(api_key=GEMINI_KEYS[current_key])
    model = genai.GenerativeModel("gemini-2.5-flash")
    print(f"\n🔄 Switched to Gemini key #{current_key+1}")


TOPICS = [

   "data visualization",
    "business intelligence",
    "predictive analytics",
    "reinforcement learning",
    "conversational AI",
    "AI ethics",
    "AI governance",
    "natural language understanding",
    "speech recognition",
    "image recognition",
    "autonomous vehicles",
    "self-driving cars",
    "wearable technology",
    "smart cities",
    "digital twins",
    "5G technology",
    "network automation",
    "cloud security",
    "cyber resilience",
    "penetration testing",
    "ethical hacking",
    "API development",
    "API security",
    "open source software",
    "version control",
    "continuous integration",
    "continuous delivery",
    "software testing",
    "agile methodology",
    "scrum practices",
    "data lakes",
    "data warehouses",
    "ETL pipelines",
    "data pipelines",
    "AI chips",
    "GPU computing",
    "parallel computing",
    "high performance computing",
    "server management",
    "IT infrastructure",
    "enterprise software",
    "SaaS platforms",
    "PaaS platforms",
    "IaaS platforms",
    "cyber physical systems",
    "digital forensics",
    "threat intelligence",
    "malware analysis",
    "ransomware defense",
    "cloud migration",
    "hybrid cloud",
    "multi cloud",
    "container orchestration",
    "DevSecOps",
    "data privacy",
    "GDPR compliance",
    "HIPAA compliance",
    "medical AI",
    "bioinformatics",
    "computational biology"
]

ARTICLES_PER_TOPIC = 50
OUTPUT_JSON = "skills_dataset.json"


# -------------------------
# Utility Functions
# -------------------------
def clean_text(text):
    if not text:
        return ""
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip()


def get_article_text(url):
    try:
        a = Article(url)
        a.download()
        a.parse()
        return clean_text(a.text)
    except:
        return ""


def parse_json_skills(s):
    s = s.strip()
    if not s:
        return []

    s = re.sub(r"```json|```", "", s)

    try:
        arr = json.loads(s)
        return [x.strip() for x in arr if x.strip()]
    except:
        m = re.search(r"\[.*?\]", s, re.DOTALL)
        if m:
            try:
                arr = json.loads(m.group())
                return [x.strip() for x in arr if x.strip()]
            except:
                return []
        return []


def extract_skills(text, retries=2):
    global model

    prompt = f"""
Extract ONLY technical skills, tools, platforms, programming languages,
frameworks, cybersecurity terms, libraries, and technologies explicitly
mentioned in the text.

Rules:
- Include programming languages (Python, Java, Go, Rust, etc.)
- Include frameworks (React, Django, TensorFlow, etc.)
- Include tools (Docker, GitHub, Kubernetes)
- Include cloud (AWS, Azure, GCP)
- Include databases (MySQL, PostgreSQL, MongoDB)
- Do NOT include soft skills
- Do NOT create new skills
- Return ONLY a pure JSON array: ["skill1", "skill2"]

Text:
{text}

Return format: ["skill1", "skill2", "skill3"]
"""

    delay = 2
    for attempt in range(retries):
        try:
            response = model.generate_content(prompt)
            return parse_json_skills(response.text)

        except Exception as e:
            e_str = str(e).lower()
            print(f"⚠️ Gemini error: {e}")

            if "quota" in e_str or "429" in e_str or "rate" in e_str:
                switch_key()
                time.sleep(delay)
                continue

            time.sleep(delay)
            delay *= 2

    return []


def improved_tokenize(text):
    return re.findall(r"\b[\w'-]+\b|[.,!?;:()\[\]{}]", text)


def tokenize_and_label(text, skills):
    tokens = improved_tokenize(text)
    labels = ["O"] * len(tokens)

    tokens_lower = [t.lower() for t in tokens]

    for skill in skills:
        skill_tokens = improved_tokenize(skill.lower())
        n = len(skill_tokens)

        for i in range(len(tokens_lower) - n + 1):
            if tokens_lower[i:i+n] == skill_tokens:
                if labels[i] == "O":
                    labels[i] = "B-SKILL"
                    for j in range(1, n):
                        if labels[i+j] == "O":
                            labels[i+j] = "I-SKILL"

    return tokens, labels


def create_article_entry(title, summary, full_text, skills, topic):
    combined_text = f"{title}. {full_text}" if full_text else f"{title}. {summary}"

    tokens, labels = tokenize_and_label(combined_text, skills)

    return {
        "topic": topic,
        "title": title,
        "summary": summary,
        "tokens": tokens,
        "ner_tags": labels,
        "extracted_skills": skills,
        "num_tokens": len(tokens),
        "num_skills_found": sum(1 for l in labels if l.startswith("B-"))
    }


# -------------------------
# Main Loop
# -------------------------
dataset = []

print("🚀 Starting data collection...")

for topic in TOPICS:
    print(f"\n📰 Fetching articles for: {topic}")

    url = f"https://gnews.io/api/v4/search?q={topic}&token={GNEWS_API_KEY}&lang=en&max={ARTICLES_PER_TOPIC}"

    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        data = r.json()
    except Exception as e:
        print(f"❌ Error fetching articles: {e}")
        continue

    articles = data.get("articles", [])
    print(f"Found {len(articles)} articles")

    for art in tqdm(articles, desc=f"Processing {topic}"):
        title = art.get("title", "")
        summary = art.get("description", "")
        link = art.get("url", "")

        full_text = get_article_text(link)
        if not full_text:
            full_text = summary

        skills = extract_skills(f"{title}. {full_text}")

        if not skills:
            continue

        entry = create_article_entry(title, summary, full_text, skills, topic)
        dataset.append(entry)

        time.sleep(0.1)


# -------------------------
# Save only Grouped CSV
# -------------------------
df_grouped = pd.DataFrame([
    {
        "topic": e["topic"],
        "title": e["title"],
        "tokens": json.dumps(e["tokens"], ensure_ascii=False),
        "ner_tags": json.dumps(e["ner_tags"], ensure_ascii=False),
        "extracted_skills": json.dumps(e["extracted_skills"], ensure_ascii=False),
        "num_tokens": e["num_tokens"],
        "num_skills_found": e["num_skills_found"]
    }
    for e in dataset
])

df_grouped.to_csv("skills_dataset_grouped.csv", index=False)

print("\n✨ Data collection complete!")
print("📁 Saved: skills_dataset_grouped.csv")


🚀 Starting data collection...

📰 Fetching articles for: data visualization
Found 8 articles


Processing data visualization:  12%|█▎        | 1/8 [00:05<00:36,  5.15s/it]WARNING:tornado.access:429 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 861.63ms


⚠️ Gemini error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 250, model: gemini-2.5-flash
Please retry in 16.571186129s.

🔄 Switched to Gemini key #2


Processing data visualization:  25%|██▌       | 2/8 [00:17<00:56,  9.46s/it]WARNING:tornado.access:429 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 810.06ms


⚠️ Gemini error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 250, model: gemini-2.5-flash
Please retry in 4.207813361s.

🔄 Switched to Gemini key #3


...

Processing computational biology:  75%|███████▌  | 3/4 [00:20<00:06,  6.65s/it]WARNING:tornado.access:429 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 962.75ms


⚠️ Gemini error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 250, model: gemini-2.5-flash
Please retry in 16.373780767s.

🔄 Switched to Gemini key #1


⚠️ Gemini error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 250, model: gemini-2.5-flash
Please retry in 13.367716212s.

🔄 Switched to Gemini key #2


Processing computational biology: 100%|██████████| 4/4 [00:26<00:00,  6.67s/it]


✨ Data collection complete!
📁 Saved: skills_dataset_grouped.csv


In [ ]:
import feedparser
from newspaper import Article
import time
import json
import re

# -------------------------------------
# USE YOUR EXISTING FUNCTIONS HERE
# extract_skills()
# tokenize_and_label()
# create_article_entry()
# -------------------------------------

GNEWS_API_KEY = ""

GEMINI_KEYS = [
    "",
    "",
    "",
    "",
    "",
    ""
]

current_key = 0

genai.configure(api_key=GEMINI_KEYS[current_key])
model = genai.GenerativeModel("gemini-2.5-flash")

def switch_key():
    global current_key, model
    current_key = (current_key + 1) % len(GEMINI_KEYS)
    genai.configure(api_key=GEMINI_KEYS[current_key])
    model = genai.GenerativeModel("gemini-2.5-flash")
    print(f"\n🔄 Switched to Gemini key #{current_key+1}")

RSS_FEEDS = {


     # =====================
    # TECH / GENERAL DEVELOPMENT
    # =====================
    "tech_news": [
        "https://medium.com/feed/topic/technology",
        "https://hackernoon.com/feed",
        "https://www.reddit.com/r/programming/.rss",
        "https://www.reddit.com/r/webdev/.rss",
        "https://medium.com/feed/topic/programming"
    ],

    # =====================
    # DEV / DEVOPS
    # =====================
    "devops_programming": [
        "https://www.reddit.com/r/devops/.rss",
        "https://www.reddit.com/r/webdev/.rss",
        "https://www.hashicorp.com/blog/rss",
        "https://www.terraform.io/blog/feed.xml"
    ],

    # =====================
    # SECURITY
    # =====================
    "security_heavy": [
        "https://www.securityweek.com/rss",
        "https://www.schneier.com/blog/atom.xml",
        "https://owasp.org/feed/"
    ],

    # =====================
    # AI / ML / RESEARCH
    # =====================
    "ai_research": [
        "https://towardsdatascience.com/feed",
        "https://www.analyticsvidhya.com/blog/feed/",
        "https://www.deeplearning.ai/blog/feed/",
        "https://www.aitrends.com/feed/",
        "https://arxiv.org/rss/cs.CL",
        "https://arxiv.org/rss/cs.CV",
        "https://arxiv.org/rss/cs.NE"
    ],
}


def extract_skills(text, retries=2):
    global model

    prompt = f"""
Extract ONLY technical skills, tools, platforms, programming languages,
frameworks, cybersecurity terms, libraries, and technologies explicitly
mentioned in the text.

Rules:
- Include programming languages (Python, Java, Go, Rust, etc.)
- Include frameworks (React, Django, TensorFlow, etc.)
- Include tools (Docker, GitHub, Kubernetes)
- Include cloud (AWS, Azure, GCP)
- Include databases (MySQL, PostgreSQL, MongoDB)
- Do NOT include soft skills
- Do NOT create new skills
- Return ONLY a pure JSON array: ["skill1", "skill2"]

Text:
{text}

Return format: ["skill1", "skill2", "skill3"]
"""

    delay = 2
    for attempt in range(retries):
        try:
            response = model.generate_content(prompt)
            return parse_json_skills(response.text)

        except Exception as e:
            e_str = str(e).lower()
            print(f"⚠️ Gemini error: {e}")

            if "quota" in e_str or "429" in e_str or "rate" in e_str:
                switch_key()
                time.sleep(delay)
                continue

            time.sleep(delay)
            delay *= 2

    return []

def tokenize_and_label(text, skills):
    tokens = improved_tokenize(text)
    labels = ["O"] * len(tokens)

    tokens_lower = [t.lower() for t in tokens]

    for skill in skills:
        skill_tokens = improved_tokenize(skill.lower())
        n = len(skill_tokens)

        for i in range(len(tokens_lower) - n + 1):
            if tokens_lower[i:i+n] == skill_tokens:
                if labels[i] == "O":
                    labels[i] = "B-SKILL"
                    for j in range(1, n):
                        if labels[i+j] == "O":
                            labels[i+j] = "I-SKILL"

    return tokens, labels


def create_article_entry(title, summary, full_text, skills, topic):
    combined_text = f"{title}. {full_text}" if full_text else f"{title}. {summary}"

    tokens, labels = tokenize_and_label(combined_text, skills)

    return {
        "topic": topic,
        "title": title,
        "summary": summary,
        "tokens": tokens,
        "ner_tags": labels,
        "extracted_skills": skills,
        "num_tokens": len(tokens),
        "num_skills_found": sum(1 for l in labels if l.startswith("B-"))
    }


def clean_text(text):
    if not text:
        return ""
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip()



def fetch_rss_articles(max_per_feed=20):
    articles = []

    print("\n📡 Starting RSS scraping...")

    for category, urls in RSS_FEEDS.items():
        print(f"\n📂 Category: {category}")

        for feed_url in urls:
            print(f"🔗 Fetching: {feed_url}")

            try:
                parsed = feedparser.parse(feed_url)
            except Exception as e:
                print(f"❌ Failed to parse {feed_url}: {e}")
                continue

            entries = parsed.entries[:max_per_feed]
            print(f"→ Found {len(entries)} items")

            for entry in entries:
                url = entry.get("link", "")
                title = entry.get("title", "")
                summary = clean_text(entry.get("summary", ""))

                # --------------------------
                # Fetch Full Article
                # --------------------------
                full_text = ""
                try:
                    a = Article(url)
                    a.download()
                    a.parse()
                    full_text = clean_text(a.text)
                except:
                    full_text = summary  # fallback

                # --------------------------
                # Skill extraction
                # --------------------------
                text_for_ai = f"{title}. {full_text}"
                skills = extract_skills(text_for_ai)

                if not skills:
                    continue

                # --------------------------
                # Build dataset entry
                # --------------------------
                entry_data = create_article_entry(
                    title=title,
                    summary=summary,
                    full_text=full_text,
                    skills=skills,
                    topic=category,   # ⭐ FIXED
                )

                articles.append(entry_data)

                time.sleep(5)

    print(f"\n📡 RSS scraping completed! Total extracted: {len(articles)}")
    return articles




In [ ]:
rss_dataset = fetch_rss_articles(max_per_feed=20)

# Save grouped CSV
df_rss_grouped = pd.DataFrame([
    {
        "topic": e["topic"],
        "title": e["title"],
        "tokens": json.dumps(e["tokens"], ensure_ascii=False),
        "ner_tags": json.dumps(e["ner_tags"], ensure_ascii=False),
        "extracted_skills": json.dumps(e["extracted_skills"], ensure_ascii=False),
        "num_tokens": e["num_tokens"],
        "num_skills_found": e["num_skills_found"]
    }
    for e in rss_dataset
])

df_rss_grouped.to_csv("rss_skills_dataset_grouped.csv", index=False)

print("\n📁 Saved: rss_skills_dataset_grouped.csv")



📡 Starting RSS scraping...

📂 Category: tech_news
🔗 Fetching: https://medium.com/feed/topic/technology
→ Found 10 items


⚠️ Gemini error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 250, model: gemini-2.5-flash
Please retry in 36.285936922s.

🔄 Switched to Gemini key #2
🔗 Fetching: https://hackernoon.com/feed
→ Found 20 items


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1543.28ms


🔗 Fetching: https://www.reddit.com/r/programming/.rss
→ Found 20 items
🔗 Fetching: https://www.reddit.com/r/webdev/.rss
→ Found 20 items
🔗 Fetching: https://medium.com/feed/topic/programming
→ Found 10 items

📂 Category: devops_programming
🔗 Fetching: https://www.reddit.com/r/devops/.rss
→ Found 20 items
🔗 Fetching: https://www.reddit.com/r/webdev/.rss
→ Found 20 items


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1919.52ms


🔗 Fetching: https://www.hashicorp.com/blog/rss
→ Found 0 items
🔗 Fetching: https://www.terraform.io/blog/feed.xml
→ Found 0 items

📂 Category: security_heavy
🔗 Fetching: https://www.securityweek.com/rss
→ Found 10 items
🔗 Fetching: https://www.schneier.com/blog/atom.xml
→ Found 10 items
🔗 Fetching: https://owasp.org/feed/
→ Found 0 items

📂 Category: ai_research
🔗 Fetching: https://towardsdatascience.com/feed
→ Found 20 items
🔗 Fetching: https://www.analyticsvidhya.com/blog/feed/
→ Found 0 items
🔗 Fetching: https://www.deeplearning.ai/blog/feed/
→ Found 0 items
🔗 Fetching: https://www.aitrends.com/feed/
→ Found 10 items
🔗 Fetching: https://arxiv.org/rss/cs.CL
→ Found 20 items
🔗 Fetching: https://arxiv.org/rss/cs.CV
→ Found 20 items
🔗 Fetching: https://arxiv.org/rss/cs.NE
→ Found 7 items

📡 RSS scraping completed! Total extracted: 190

📁 Saved: rss_skills_dataset_grouped.csv
